# 04 - Gold Layer – Business Analytics

## Procurement Analytics Pipeline

The Gold Layer contains business-ready analytical datasets generated from curated Silver layer data.

### Objectives

Generate procurement insights such as:

* Total vendor spend
* Invoice vs contract price variance
* Vendor risk classification
* Region-wise spend analysis
* Monthly procurement spend trends

These outputs are designed for Power BI dashboarding and business reporting.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, avg, when, month, year, concat_ws
import os

## Initialize Spark Session

Create a Spark session for Gold layer analytics.


In [5]:
spark = SparkSession.builder \
    .appName('Gold Analytics') \
    .master('local[*]') \
    .getOrCreate()

print('Spark Started')

Spark Started


## Load Silver Layer Datasets

Load the curated Silver datasets generated in previous notebooks.


In [6]:
orders = spark.read.option('header', True).csv('output/silver/orders_silver.csv')

invoices = spark.read.option('header', True).csv('output/silver/invoices_silver.csv')

vendors = spark.read.option('header', True).csv('output/silver/vendors_silver.csv')

contracts = spark.read.option('header', True).csv('output/silver/scd/current_contracts.csv')

print('Silver datasets loaded successfully')

Silver datasets loaded successfully


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_replace, col

spark = SparkSession.builder.master('local[*]').appName('fix').getOrCreate()

# Reload fresh data from CSV
orders = spark.read.option('header', True).csv('output/silver/orders_silver.csv')
invoices = spark.read.option('header', True).csv('output/silver/invoices_silver.csv')
vendors = spark.read.option('header', True).csv('output/silver/vendors_silver.csv')
contracts = spark.read.option('header', True).csv('output/silver/scd/current_contracts.csv')

# Clean numeric columns
orders = orders.withColumn('quantity_requested', col('quantity_requested').cast('int'))

invoices = invoices.withColumn(
    'invoiced_price_per_unit',
    regexp_replace(col('invoiced_price_per_unit'), '\\$', '').cast('double')
)

contracts = contracts.withColumn(
    'negotiated_price',
    regexp_replace(col('negotiated_price'), '\\$', '').cast('double')
)

# Join
base_df = orders.alias('o') \
    .join(invoices.alias('i'), 'po_id') \
    .join(vendors.alias('v'), 'vendor_id') \
    .join(contracts.alias('c'), ['vendor_id', 'item_name'], 'left')

# Metrics
analytics_df = base_df \
    .withColumn('invoice_amount', col('quantity_requested') * col('invoiced_price_per_unit')) \
    .withColumn('contract_amount', col('quantity_requested') * col('negotiated_price')) \
    .withColumn('price_variance', col('invoice_amount') - col('contract_amount'))

analytics_df.select(
    'po_id',
    'vendor_id',
    'item_name',
    'invoice_amount',
    'contract_amount',
    'price_variance'
).show(5, truncate=False)

+---------+---------+----------+------------------+------------------+-------------------+
|po_id    |vendor_id|item_name |invoice_amount    |contract_amount   |price_variance     |
+---------+---------+----------+------------------+------------------+-------------------+
|PO0000003|V03667   |Models    |86566.56          |35565.840000000004|51000.719999999994 |
|PO0000003|V03667   |Models    |72802.79999999999 |35565.840000000004|37236.959999999985 |
|PO0000006|V01322   |E-services|1475809.44        |4178194.7199999997|-2702385.28        |
|PO0000006|V01322   |E-services|3828115.7600000002|4178194.7199999997|-350078.9599999995 |
|PO0000006|V01322   |E-services|1288827.1199999999|4178194.7199999997|-2889367.5999999996|
+---------+---------+----------+------------------+------------------+-------------------+
only showing top 5 rows


# Business Insight 1 – Total Vendor Spend

Calculate total procurement spend for each vendor.


In [8]:
vendor_spend = analytics_df.groupBy(
    'vendor_id',
    'vendor_name'
).agg(
    _sum('invoice_amount').alias('total_spend')
).orderBy(
    col('total_spend').desc()
)

vendor_spend.show(10, truncate=False)

+---------+-----------------+--------------------+
|vendor_id|vendor_name      |total_spend         |
+---------+-----------------+--------------------+
|V04381   |Price-Dominguez  |2.562041008E7       |
|V01160   |Perez Ltd        |2.2728855189999998E7|
|V01817   |English-Dominguez|2.1949849949999996E7|
|V01347   |Keith-Williams   |2.1413345909999996E7|
|V00400   |Duran LLC        |2.0661464990000002E7|
|V02660   |Hale Inc         |2.000366798E7       |
|V03780   |Martin-Payne     |1.971023083E7       |
|V04456   |elliott-zamora   |1.939729098E7       |
|V04639   |Goodman and Sons |1.782040627E7       |
|V00162   |Stein-Silva      |1.7383200650000002E7|
+---------+-----------------+--------------------+
only showing top 10 rows


# Business Insight 2 – Invoice vs Contract Price Variance

Identify vendors where invoiced prices differ from negotiated contract prices.


In [9]:
price_variance = analytics_df.groupBy(
    'vendor_id',
    'vendor_name'
).agg(
    _sum('price_variance').alias('total_price_variance')
).orderBy(
    col('total_price_variance').desc()
)

price_variance.show(10, truncate=False)

+---------+---------------------------+--------------------+
|vendor_id|vendor_name                |total_price_variance|
+---------+---------------------------+--------------------+
|V01475   |Edwards Ltd                |2.295532975E7       |
|V02338   |Leonard, Ballard and Miller|1.8354878760000005E7|
|V02183   |Hayden, Harmon and Mcdonald|1.770751488E7       |
|V01123   |Guerrero-Becker            |1.606700389E7       |
|V04008   |Mcdonald-Murray            |1.5039330880000003E7|
|V04274   |Moore PLC                  |1.4547691989999998E7|
|V01774   |Callahan PLC               |1.4459809549999999E7|
|V00066   |Morton-Chase               |1.426590217E7       |
|V00748   |Goodwin Ltd                |1.3010015870000001E7|
|V04995   |Huang, Hall and Delacruz   |1.217577513E7       |
+---------+---------------------------+--------------------+
only showing top 10 rows


# Business Insight 3 – Vendor Risk Classification

Classify vendors into High, Medium, and Low risk categories.


In [11]:
vendor_risk = vendors.select(
    'vendor_id',
    'vendor_name',
    'risk_rating'
).withColumnRenamed(
    'risk_rating',
    'risk_category'
)

vendor_risk.groupBy('risk_category').count().show()

+-------------+-----+
|risk_category|count|
+-------------+-----+
|         High| 1566|
|          Low| 1580|
|         NULL|  247|
|       Medium| 1607|
+-------------+-----+



# Business Insight 4 – Region-wise Spend Analysis

Calculate procurement spend by vendor region.


In [12]:
region_spend = analytics_df.groupBy('region').agg(
    _sum('invoice_amount').alias('region_total_spend')
).orderBy(
    col('region_total_spend').desc()
)

region_spend.show(truncate=False)

+------+--------------------+
|region|region_total_spend  |
+------+--------------------+
|AMER  |1.619774159080001E9 |
|EMEA  |1.5040599929800007E9|
|APAC  |1.4878802566400013E9|
|LATAM |1.4052269322199988E9|
+------+--------------------+



# Business Insight 5 – Monthly Procurement Spend Trend

Analyze procurement spend across months.


In [13]:
monthly_spend = analytics_df.withColumn(
    'po_timestamp',
    col('po_timestamp').cast('timestamp')
).withColumn(
    'year',
    year(col('po_timestamp'))
).withColumn(
    'month',
    month(col('po_timestamp'))
).groupBy(
    'year',
    'month'
).agg(
    _sum('invoice_amount').alias('monthly_spend')
).orderBy('year', 'month')

monthly_spend.show(truncate=False)

+----+-----+--------------------+
|year|month|monthly_spend       |
+----+-----+--------------------+
|2024|4    |1.7350897078E8      |
|2024|5    |2.426185386799999E8 |
|2024|6    |2.1514897310999998E8|
|2024|7    |2.6441088285999992E8|
|2024|8    |2.5637837514999983E8|
|2024|9    |2.2065159100000012E8|
|2024|10   |2.5265568154999992E8|
|2024|11   |2.4712224318000007E8|
|2024|12   |2.4733789133999997E8|
|2025|1    |2.6585725141000006E8|
|2025|2    |2.2181806534999996E8|
|2025|3    |2.744561809499999E8 |
|2025|4    |2.3362417254999995E8|
|2025|5    |2.0871624873000005E8|
|2025|6    |2.8570295857000005E8|
|2025|7    |3.0277895252999985E8|
|2025|8    |2.7054547624E8      |
|2025|9    |1.797837034400001E8 |
|2025|10   |2.5235028261000007E8|
|2025|11   |2.2884673482999992E8|
+----+-----+--------------------+
only showing top 20 rows


## Create Power BI Ready Tables

Save Gold analytical outputs as CSV files.


In [14]:
os.makedirs('output/gold', exist_ok=True)

vendor_spend.toPandas().to_csv(
    'output/gold/vendor_spend.csv',
    index=False
)

price_variance.toPandas().to_csv(
    'output/gold/price_variance.csv',
    index=False
)

vendor_risk.toPandas().to_csv(
    'output/gold/vendor_risk.csv',
    index=False
)

region_spend.toPandas().to_csv(
    'output/gold/region_spend.csv',
    index=False
)

monthly_spend.toPandas().to_csv(
    'output/gold/monthly_spend.csv',
    index=False
)

print('Gold layer files saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use

Gold layer files saved successfully


## Verify Gold Files

Confirm that all Gold analytical datasets were generated successfully.


In [15]:
print(os.listdir('output/gold'))

['monthly_spend.csv', 'price_variance.csv', 'region_spend.csv', 'vendor_risk.csv', 'vendor_spend.csv']


## Gold Layer Summary

| Analytical Table | Purpose                       |
| ---------------- | ----------------------------- |
| vendor_spend     | Vendor spend analysis         |
| price_variance   | Invoice vs contract variance  |
| vendor_risk      | Vendor risk categorization    |
| region_spend     | Regional procurement analysis |
| monthly_spend    | Monthly spend trend analysis  |


# Conclusion

The Gold Layer successfully transformed curated procurement data into business-ready analytical datasets. Vendor spend, contract variance, risk classification, regional spend, and monthly procurement trends were generated and exported for Power BI dashboarding.

This notebook completes the Medallion Architecture pipeline by delivering actionable procurement insights for business users.
